# Guardrail Hooks: Blocking Unsafe Outputs in Real-Time

Based on: [StepShield: When, Not Whether to Intervene](https://arxiv.org/abs/2601.22136) (Jan 2026)

## The Problem

Post-hoc safety evaluation (Demos 01 and 02) catches problems *after* the user sees the response. In production, you need to **block** unsafe outputs *before* delivery.

## The Technique: `event.cancel_tool`

A Strands `HookProvider` intercepts `BeforeToolCallEvent` and decides whether to allow or block the tool call. The mechanism is straightforward:

```python
def _check(self, event: BeforeToolCallEvent):
    if tool_is_dangerous(event.tool_use["name"]):
        # Block the tool — it will NOT execute
        event.cancel_tool = "Reason for blocking"
    # If cancel_tool is not set, the tool executes normally
```

When `event.cancel_tool` is set to a string, three things happen:
1. The tool **does not execute** — no side effects, no API calls
2. The cancellation reason string is returned to the agent as the tool's "result"
3. The agent can then respond to the user explaining why the action was blocked

This is different from raising an exception: the agent continues running and can gracefully explain the block to the user, rather than crashing.

| Guardrail Type | What It Blocks | Implementation |
|---------------|---------------|----------------|
| Tool blocklist | Prevent specific tools from executing | Check tool name against a list, set `event.cancel_tool` |
| PII filter | Block responses containing PII | Regex on model output in `AfterModelCallEvent` |
| Policy check | Block actions violating business rules | Custom logic per tool, inspect parameters |

> **What to look for:** The demo creates a `SafetyGuardrail` that blocks `cancel_booking` and `book_hotel`. Safe queries (flight search) should execute normally. Dangerous queries (booking, cancellation) should be blocked with a clear reason.

In [ ]:
# %pip install strands-agents strands-agents-evals boto3

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

import sys
sys.path.insert(0, "../../evaluate-tool-use/01-tool-selection-accuracy")

from strands import Agent
from strands.models.openai import OpenAIModel
from strands.hooks import HookProvider, HookRegistry
from strands.hooks.events import BeforeToolCallEvent
from travel_tools import ALL_TOOLS

MODEL = "gpt-4o-mini"


class SafetyGuardrail(HookProvider):
    """Blocks dangerous tool calls before execution.

    - Blocks tools in the blocklist
    - Logs all blocked and allowed calls
    """

    def __init__(self, blocked_tools: list[str] = None):
        self.blocked_tools = blocked_tools or ["cancel_booking", "book_hotel"]
        self.log = []

    def register_hooks(self, registry: HookRegistry, **kwargs) -> None:
        registry.add_callback(BeforeToolCallEvent, self._check)

    def _check(self, event: BeforeToolCallEvent) -> None:
        tool_name = event.tool_use["name"]
        if tool_name in self.blocked_tools:
            event.cancel_tool = f"BLOCKED by SafetyGuardrail: {tool_name} requires explicit approval"
            self.log.append({"tool": tool_name, "action": "BLOCKED", "params": event.tool_use.get("input", {})})
            print(f"  🛑 BLOCKED: {tool_name} (requires approval)")
        else:
            self.log.append({"tool": tool_name, "action": "ALLOWED"})
            print(f"  ✅ ALLOWED: {tool_name}")


guardrail = SafetyGuardrail(blocked_tools=["cancel_booking", "book_hotel"])

agent = Agent(
    model=OpenAIModel(model_id=MODEL),
    tools=ALL_TOOLS,
    hooks=[guardrail],
    system_prompt="You are a travel assistant. Use tools to answer questions.",
)

print("=" * 60)
print("GUARDRAIL DEMO: Blocking dangerous tool calls")
print(f"Blocked tools: {guardrail.blocked_tools}")
print("=" * 60)

# Query 1: Safe — should execute normally
print("\n--- Query: 'Find flights NYC to London' ---")
agent("Find flights NYC to London for next Friday")

# Query 2: Dangerous — book_hotel should be blocked
print("\n--- Query: 'Book the Marriott for Alex, March 20-22' ---")
agent("Book the Marriott for me, March 20-22")

# Query 3: Dangerous — cancel_booking should be blocked
print("\n--- Query: 'Cancel my booking H12345' ---")
agent("Cancel my booking H12345")

# Summary
print(f"\n📊 Guardrail Summary:")
blocked = [l for l in guardrail.log if l["action"] == "BLOCKED"]
allowed = [l for l in guardrail.log if l["action"] == "ALLOWED"]
print(f"   Allowed: {len(allowed)} tool calls")
print(f"   Blocked: {len(blocked)} tool calls")
for b in blocked:
    print(f"   🛑 {b['tool']}({b.get('params', {})})")

## Series Summary: 3 Layers of Safety Evaluation

**What to look for in the results above:** Query 1 (flight search) should show all tool calls allowed. Query 2 (book hotel) should show `book_hotel` blocked. Query 3 (cancel booking) should show `cancel_booking` blocked. The agent should respond gracefully to blocked calls, explaining that the action requires approval.

| Layer | Demo | What It Does | When | Cost |
|-------|------|-------------|------|:----:|
| 1. Scoring | 01 | Score responses for safety (PII, harmful content) | After execution | Free + 1 call |
| 2. Drift | 02 | Track safety across conversation turns | After conversation | N calls |
| 3. Guardrails | 03 | Block unsafe tool calls before execution | During execution | Free |

**Use all 3:** Layer 3 blocks known-dangerous actions in real-time (free, instant). Layer 1 catches semantic safety issues after the response is generated. Layer 2 monitors long conversations for gradual degradation that individual checks miss.